In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [2]:
n_hidden = 35 # 순환 신경망의 노드 수 = 은닉 상태 벡터의 길이. 클수록 과거 정보를 많이 담을 수 있음.
lr = 0.01
epochs = 1000

string = "hello pytorch. how long can a rnn cell remember?" # show us your linmit"
chars = "abcdefghijklmnopqrstuvwxyz ?!.,:;01"
char_list = [i for i in chars]
n_letters = len(char_list)

In [3]:
def string_to_onehot(string):
    start = np.zeros(shape=len(char_list), dtype=int)
    end = np.zeros(shape=len(char_list), dtype=int)
    start[-2] = 1
    end[-1] = 1
    for i in string:
        idx = char_list.index(i)
        zero = np.zeros(shape=n_letters, dtype=int)
        zero[idx] = 1
        start = np.vstack([start, zero])
    
    output = np.vstack([start, end])
    return output

In [4]:
def onehot_to_word(onehot_1):
    onehot = torch.Tensor.numpy(onehot_1)
    return char_list[onehot.argmax()]

In [5]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()

        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size

        self.i2h = nn.Linear(input_size, hidden_size)
        self.h2h = nn.Linear(hidden_size, hidden_size)
        self.i2o = nn.Linear(hidden_size, output_size)
        self.act_fn = nn.Tanh()

    def forward(self, input, hidden):
        hidden = self.act_fn(self.i2h(input)+self.h2h(hidden))
        output = self.i2o(hidden)
        return output, hidden

    def init_hidden(self):
        return torch.zeros(1, self.hidden_size)

rnn = RNN(n_letters, n_hidden, n_letters)

In [6]:
loss_func = nn.MSELoss()
optimizer = torch.optim.Adam(rnn.parameters(), lr=lr)

In [7]:
one_hot = torch.from_numpy(string_to_onehot(string)).type_as(torch.FloatTensor())

for i in range(epochs):
    rnn.zero_grad()
    total_loss = 0
    hidden = rnn.init_hidden()

    for j in range(one_hot.size()[0] - 1):
        input_ = one_hot[j:j+1,:]
        target = one_hot[j+1]

        output, hidden = rnn.forward(input_, hidden)
        loss = loss_func(output.view(-1), target.view(-1))
        total_loss += loss
        input_ = output

    total_loss.backward()
    optimizer.step()

    if i % 10 == 0:
        print(total_loss)

tensor(2.6949, grad_fn=<AddBackward0>)
tensor(1.0182, grad_fn=<AddBackward0>)
tensor(0.6978, grad_fn=<AddBackward0>)
tensor(0.4533, grad_fn=<AddBackward0>)
tensor(0.3013, grad_fn=<AddBackward0>)
tensor(0.2294, grad_fn=<AddBackward0>)
tensor(0.1848, grad_fn=<AddBackward0>)
tensor(0.1433, grad_fn=<AddBackward0>)
tensor(0.1144, grad_fn=<AddBackward0>)
tensor(0.1203, grad_fn=<AddBackward0>)
tensor(0.0861, grad_fn=<AddBackward0>)
tensor(0.0728, grad_fn=<AddBackward0>)
tensor(0.0631, grad_fn=<AddBackward0>)
tensor(0.0648, grad_fn=<AddBackward0>)
tensor(0.0547, grad_fn=<AddBackward0>)
tensor(0.0479, grad_fn=<AddBackward0>)
tensor(0.0490, grad_fn=<AddBackward0>)
tensor(0.0417, grad_fn=<AddBackward0>)
tensor(0.0356, grad_fn=<AddBackward0>)
tensor(0.0318, grad_fn=<AddBackward0>)
tensor(0.0377, grad_fn=<AddBackward0>)
tensor(0.0290, grad_fn=<AddBackward0>)
tensor(0.0260, grad_fn=<AddBackward0>)
tensor(0.0288, grad_fn=<AddBackward0>)
tensor(0.0230, grad_fn=<AddBackward0>)
tensor(0.0248, grad_fn=<A

In [8]:
start = torch.zeros(1, len(char_list))
start[:, -2] = 1

with torch.no_grad():
    hidden = rnn.init_hidden()
    input_ = start
    output_string = ""
    for i in range(len(string)):
        output, hidden = rnn.forward(input_, hidden)
        output_string += onehot_to_word(output.data)
        input_ = output

print(output_string)

hello crpreo c  ce rnrr  ce rn cewrc pe re rewrl
